# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import requests
import pandas as pd

# Official Census Building Permits resources
sources = {
    "Annual Metropolitan Data":
        "https://www.census.gov/construction/bps/msaannual.html",
    "Historical Metropolitan Units":
        "https://www.census.gov/construction/bps/historical/metro_units.html",
    "Current Building Permits Data":
        "https://www.census.gov/construction/bps/current.html"
}

# Test whether each page is accessible
for name, url in sources.items():
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        print(f"{name}")
        print(f"Status Code: {response.status_code}")
        print(f"Content Type: {response.headers.get('Content-Type')}")
        print("-" * 50)

    except requests.RequestException as error:
        print(f"{name}: Connection error - {error}")
        print("-" * 50)

Annual Metropolitan Data
Status Code: 200
Content Type: text/html; charset=utf-8
--------------------------------------------------
Historical Metropolitan Units
Status Code: 200
Content Type: text/html; charset=utf-8
--------------------------------------------------
Current Building Permits Data
Status Code: 200
Content Type: text/html; charset=utf-8
--------------------------------------------------


In [2]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

url = "https://www.census.gov/construction/bps/msaannual.html"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Identify downloadable files linked on the page.
file_links = []

for anchor in soup.find_all("a", href=True):
    href = anchor["href"]
    full_url = urljoin(url, href)

    if re.search(r"\.(xls|xlsx|txt)(?:\?|$)", full_url, re.I):
        file_links.append({
            "label": anchor.get_text(" ", strip=True),
            "url": full_url
        })

links_df = pd.DataFrame(file_links).drop_duplicates()

print("Downloadable file links found:", len(links_df))
display(links_df.head(30))


Downloadable file links found: 35


,label,url
0,2019,https://www.census.gov/construction/bps/xls/ms...
1,2020,https://www.census.gov/construction/bps/xls/ms...
2,2021 (Final),https://www.census.gov/construction/bps/xls/ms...
3,TXT,https://www.census.gov/construction/bps/txt/tb...
4,TXT,https://www.census.gov/construction/bps/txt/tb...
5,TXT,https://www.census.gov/construction/bps/txt/tb...
6,TXT,https://www.census.gov/construction/bps/txt/tb...
7,TXT,https://www.census.gov/construction/bps/txt/tb...
8,TXT,https://www.census.gov/construction/bps/txt/tb...
9,TXT,https://www.census.gov/construction/bps/txt/tb...


In [3]:

from pathlib import PurePosixPath
from urllib.parse import urlparse

# Extract the filename from each discovered URL.
links_df["filename"] = links_df["url"].apply(
    lambda url: PurePosixPath(urlparse(url).path).name
)

# Display all discovered files without truncating URLs.
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

display(
    links_df[["label", "filename", "url"]]
    .sort_values("filename")
    .reset_index(drop=True)
)


,label,filename,url
0,2019,msaannual_201999.xls,https://www.census.gov/construction/bps/xls/msaannual_201999.xls
1,2020,msaannual_202099.xls,https://www.census.gov/construction/bps/xls/msaannual_202099.xls
2,2021 (Final),msaannual_202199.xls,https://www.census.gov/construction/bps/xls/msaannual_202199.xls
3,TXT,tb3u2004.txt,https://www.census.gov/construction/bps/txt/tb3u2004.txt
4,TXT,tb3u2005.txt,https://www.census.gov/construction/bps/txt/tb3u2005.txt
5,TXT,tb3u2006.txt,https://www.census.gov/construction/bps/txt/tb3u2006.txt
6,TXT,tb3u2007.txt,https://www.census.gov/construction/bps/txt/tb3u2007.txt
7,TXT,tb3u2008.txt,https://www.census.gov/construction/bps/txt/tb3u2008.txt
8,TXT,tb3u2009.txt,https://www.census.gov/construction/bps/txt/tb3u2009.txt
9,TXT,tb3u2010.txt,https://www.census.gov/construction/bps/txt/tb3u2010.txt


In [4]:

# Inspect the two historical text-file series for 2018.

historical_files = {
    "2018_u": "https://www.census.gov/construction/bps/txt/tb3u2018.txt",
    "2018_v": "https://www.census.gov/construction/bps/txt/tb3v2018.txt"
}

for name, url in historical_files.items():
    print(f"\n{'=' * 65}")
    print(f"FILE: {name}")
    print(f"URL: {url}")

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    lines = response.text.splitlines()

    print(f"Total lines: {len(lines)}")
    print("\nFIRST 15 LINES:\n")

    for line in lines[:15]:
        print(line[:200])



FILE: 2018_u
URL: https://www.census.gov/construction/bps/txt/tb3u2018.txt
Total lines: 420

FIRST 15 LINES:

Table 3au. New Privately Owned Housing Units Authorized
           Unadjusted Units by Metropolitan Area
 
           Annual 2018
                                                                                        Num of
                                                                                        Struc-
                                                                                        tures
                                                                                        With
                                                                      3 and 4  5 Units  5 Units
CSA CBSA  Name                                   Total 1 Unit 2 Units   Units  or more  or more

999 10180 Abilene, TX                             357     335      22       0       0       0
184 10420 Akron, OH                              1210    1011       8      12     179       2
9

In [5]:

# Our seven selected metropolitan statistical areas
target_cbsas = {
    "16700": "Charleston–North Charleston, SC",
    "34980": "Nashville, TN",
    "35620": "New York, NY",
    "41860": "San Francisco, CA",
    "19100": "Dallas–Fort Worth, TX",
    "28140": "Kansas City, MO–KS",
    "29820": "Las Vegas, NV"
}

# Reuse the 2018 units file
url = "https://www.census.gov/construction/bps/txt/tb3u2018.txt"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

lines = response.text.splitlines()

print("2018 METROPOLITAN AREA MATCHES")
print("=" * 75)

for cbsa, name in target_cbsas.items():
    matches = [
        line for line in lines
        if re.match(rf"^\s*\d{{3}}\s+{cbsa}\s+", line)
    ]

    print(f"\n{cbsa} — {name}")
    if matches:
        for match in matches:
            print(match)
    else:
        print("NOT FOUND")


2018 METROPOLITAN AREA MATCHES

16700 — Charleston–North Charleston, SC
999 16700 Charleston-North Charleston,* SC       7002    4787       0      16    2199      77

34980 — Nashville, TN
400 34980 Nashville-Davidson--Murfreesboro--Franklin,

35620 — New York, NY
408 35620 New York-Newark-Jersey City,

41860 — San Francisco, CA
488 41860 San Francisco-Oakland-Hayward,* CA    17421    4048     154     259   12960     219

19100 — Dallas–Fort Worth, TX
206 19100 Dallas-Fort Worth-Arlington, TX       63893   36832     354     484   26223     617

28140 — Kansas City, MO–KS
312 28140 Kansas City, MO-KS                    10268    5608     258     388    4014     125

29820 — Las Vegas, NV
332 29820 Las Vegas-Henderson-Paradise,* NV     12044    9721      22      30    2271     153


In [6]:

# Inspect the lines surrounding Nashville and New York.

target_codes = ["34980", "35620"]

for cbsa in target_codes:
    print(f"\n{'=' * 75}")
    print(f"CBSA: {cbsa}")
    print("=" * 75)

    for index, line in enumerate(lines):
        if re.match(rf"^\s*\d{{3}}\s+{cbsa}\s+", line):
            start = max(0, index - 1)
            end = min(len(lines), index + 4)

            for i in range(start, end):
                print(f"Line {i + 1}: {lines[i]!r}")

            break



CBSA: 34980
Line 269: '162 34940 Naples-Immokalee-Marco Island, FL      4386    3253       8      78    1047      59'
Line 270: '400 34980 Nashville-Davidson--Murfreesboro--Franklin,'
Line 271: '  TN                                            19159   13470      22      77    5590     135'
Line 272: '404 35100 New Bern, NC                            389     389       0       0       0       0'
Line 273: '408 35300 New Haven-Milford, CT                  1166     406      48      18     694      22'

CBSA: 35620
Line 274: '406 35380 New Orleans-Metairie, LA               3864    3046     174      22     622      12'
Line 275: '408 35620 New York-Newark-Jersey City,'
Line 276: '  NY-NJ-PA                                      49692   11077    1682    1061   35872    1120'
Line 277: '515 35660 Niles-Benton Harbor, MI                 302     237       2       0      63       3'
Line 278: '412 35840 North Port-Sarasota-Bradenton,* FL    10289    6383      36      93    3777     133'


In [7]:

import re
import pandas as pd

# Identify the beginning of a metropolitan-area record.
record_pattern = re.compile(r"^\s*(\d{3})\s+(\d{5})\s+(.+)$")

records = []
current_record = None

for line in lines:

    # Stop when the Census footnote section begins.
    if line.strip().startswith(
        "* - Metropolitan areas where all permit offices"
    ):
        break

    match = record_pattern.match(line)

    if match:
        # Save the previous metropolitan record.
        if current_record is not None:
            records.append(current_record)

        current_record = {
            "csa": match.group(1),
            "cbsa": match.group(2),
            "text": match.group(3).strip()
        }

    elif current_record is not None and line.strip():
        # Append continuation lines.
        current_record["text"] += " " + line.strip()

# Save the final record.
if current_record is not None:
    records.append(current_record)

# Extract the six numeric columns from the end of each record.
parsed_records = []

for record in records:
    match = re.search(
        r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
        record["text"]
    )

    if match:
        parsed_records.append({
            "csa": record["csa"],
            "cbsa": record["cbsa"],
            "metro_name": match.group(1).strip(),
            "permits_total": int(match.group(2)),
            "units_1": int(match.group(3)),
            "units_2": int(match.group(4)),
            "units_3_4": int(match.group(5)),
            "units_5_plus": int(match.group(6)),
            "structures_5_plus": int(match.group(7)),
            "year": 2018
        })

permits_2018 = pd.DataFrame(parsed_records)

# Filter to our seven selected metropolitan areas.
selected_2018 = permits_2018[
    permits_2018["cbsa"].isin(target_cbsas.keys())
].copy()

print("Total metropolitan records parsed:", len(permits_2018))
print("Selected MSA records:", len(selected_2018))
print("Unique selected CBSAs:", selected_2018["cbsa"].nunique())

display(
    selected_2018[
        ["cbsa", "metro_name", "year", "permits_total"]
    ].sort_values("cbsa")
)


Total metropolitan records parsed: 382
Selected MSA records: 7
Unique selected CBSAs: 7


,cbsa,metro_name,year,permits_total
63,16700,"Charleston-North Charleston,* SC",2018,7002
86,19100,"Dallas-Fort Worth-Arlington, TX",2018,63893
180,28140,"Kansas City, MO-KS",2018,10268
197,29820,"Las Vegas-Henderson-Paradise,* NV",2018,12044
245,34980,"Nashville-Davidson--Murfreesboro--Franklin, TN",2018,19159
249,35620,"New York-Newark-Jersey City, NY-NJ-PA",2018,49692
306,41860,"San Francisco-Oakland-Hayward,* CA",2018,17421


In [8]:

# Verify that housing-unit categories sum to the reported total.
permits_2018["calculated_total"] = (
    permits_2018["units_1"]
    + permits_2018["units_2"]
    + permits_2018["units_3_4"]
    + permits_2018["units_5_plus"]
)

permits_2018["total_difference"] = (
    permits_2018["permits_total"]
    - permits_2018["calculated_total"]
)

print("Total records:", len(permits_2018))
print(
    "Records with mismatched totals:",
    (permits_2018["total_difference"] != 0).sum()
)

display(
    permits_2018.loc[
        permits_2018["total_difference"] != 0,
        ["cbsa", "metro_name", "permits_total",
         "calculated_total", "total_difference"]
    ].head(20)
)


Total records: 382
Records with mismatched totals: 0


,cbsa,metro_name,permits_total,calculated_total,total_difference


In [9]:

# Count all apparent metropolitan records in the raw 2018 file.
record_starts = []

for line_number, line in enumerate(lines, start=1):
    match = record_pattern.match(line)

    if match:
        record_starts.append({
            "line_number": line_number,
            "cbsa": match.group(2),
            "raw_line": line
        })

print("Metropolitan record starts identified:", len(record_starts))
print("Records successfully parsed:", len(permits_2018))

print(
    "Difference:",
    len(record_starts) - len(permits_2018)
)

# Check whether any CBSA appears more than once.
duplicate_cbsas = permits_2018[
    permits_2018.duplicated(subset=["cbsa"], keep=False)
]

print("Duplicate CBSA records:", len(duplicate_cbsas))

# Confirm all seven target CBSAs are present.
missing_cbsas = (
    set(target_cbsas.keys())
    - set(permits_2018["cbsa"])
)

print("Missing target CBSAs:", sorted(missing_cbsas))


Metropolitan record starts identified: 382
Records successfully parsed: 382
Difference: 0
Duplicate CBSA records: 0
Missing target CBSAs: []


In [10]:

# Identify CBSA codes detected in the raw text
# but absent from the successfully parsed records.

raw_cbsas = {
    record["cbsa"] for record in record_starts
}

parsed_cbsas = set(permits_2018["cbsa"])

unmatched_cbsas = raw_cbsas - parsed_cbsas

print("Unmatched CBSA codes:", sorted(unmatched_cbsas))

# Display the original source lines for unmatched records.
for record in record_starts:
    if record["cbsa"] in unmatched_cbsas:
        line_number = record["line_number"]

        print(f"\nCBSA: {record['cbsa']}")
        print(f"Source line: {line_number}")
        print("Surrounding source text:")

        start = max(0, line_number - 2)
        end = min(len(lines), line_number + 3)

        for i in range(start, end):
            print(f"Line {i + 1}: {lines[i]!r}")


Unmatched CBSA codes: []


In [11]:

# Inspect the 2005 historical Building Permits file.

url_2005 = (
    "https://www.census.gov/construction/bps/txt/tb3u2005.txt"
)

response_2005 = requests.get(
    url_2005,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response_2005.raise_for_status()

lines_2005 = response_2005.text.splitlines()

print("2005 FILE INSPECTION")
print("=" * 65)
print("Total lines:", len(lines_2005))

print("\nFIRST 15 LINES:\n")

for line in lines_2005[:15]:
    print(repr(line))


2005 FILE INSPECTION
Total lines: 398

FIRST 15 LINES:

'Table 3au. New Privately Owned Housing Units Authorized'
'           Unadjusted Units by Metropolitan Area'
' '
'           Annual 2005'
'                                                                              Num of'
'                                                                              Struc-'
'                                                                              tures'
'                                                                              With'
'                                                          3 and 4  5 Units  5 Units'
'                                     Total 1 Unit 2 Units   Units  or more  or more'
''
'Abilene, TX                             346     310      22       0      14       1'
'Akron, OH                              2724    2310      98     105     211      14'
'Albany, GA                              751     689      22      32       8       1'
'Albany-Schenectady-Troy, NY

In [12]:

# Locate target metropolitan areas in the 2005 source file.
# Search by recognizable city names because CBSA codes
# are not present in the records we've inspected.

search_terms = [
    "Charleston",
    "Dallas",
    "Kansas City",
    "Las Vegas",
    "Nashville",
    "New York",
    "San Francisco"
]

print("2005 TARGET METROPOLITAN AREA INSPECTION")
print("=" * 65)

for term in search_terms:
    matches = [
        (line_number, line)
        for line_number, line in enumerate(lines_2005, start=1)
        if term.lower() in line.lower()
    ]

    print(f"\n{term}: {len(matches)} match(es)")

    for line_number, line in matches:
        print(f"  Line {line_number}: {line!r}")


2005 TARGET METROPOLITAN AREA INSPECTION

Charleston: 2 match(es)
  Line 70: 'Charleston, WV                          665     517       2       0     146       9'
  Line 71: 'Charleston-North Charleston,* SC      10818    8084     156     129    2449     155'

Dallas: 1 match(es)
  Line 94: 'Dallas-Fort Worth-Arlington,* TX      59895   49489     442     152    9812     509'

Kansas City: 1 match(es)
  Line 187: 'Kansas City,* MO-KS                   15218   12118     528     749    1823     109'

Las Vegas: 1 match(es)
  Line 203: 'Las Vegas-Paradise,* NV               39237   30479      96    1155    7507     348'

Nashville: 1 match(es)
  Line 252: 'Nashville-Davidson--Murfreesboro,*'

New York: 1 match(es)
  Line 256: 'New York-Northern New Jersey-Long Island,*'

San Francisco: 1 match(es)
  Line 322: 'San Francisco-Oakland-Fremont,* CA    14883    8102      78     294    6409     188'


In [13]:

# Examine the complete source text surrounding
# Nashville and New York in the 2005 file.

for start_line, end_line, metro in [
    (249, 255, "Nashville"),
    (254, 260, "New York")
]:
    print(f"\n{metro.upper()} — SOURCE RECORD")
    print("=" * 65)

    for line_number in range(start_line, end_line + 1):
        print(
            f"Line {line_number}: "
            f"{lines_2005[line_number - 1]!r}"
        )



NASHVILLE — SOURCE RECORD
Line 249: '  SC                                  11828    6471      84     141    5132     264'
Line 250: 'Napa, CA                                687     628       8       0      51       3'
Line 251: 'Naples-Marco Island,* FL               6622    4052     104     547    1919     149'
Line 252: 'Nashville-Davidson--Murfreesboro,*'
Line 253: '  TN                                  16654   14074     322      75    2183     123'
Line 254: 'New Haven-Milford, CT                  2251    1693      30      20     508      26'
Line 255: 'New Orleans-Metairie-Kenner,* LA       4781    4488     152      36     105      12'

NEW YORK — SOURCE RECORD
Line 254: 'New Haven-Milford, CT                  2251    1693      30      20     508      26'
Line 255: 'New Orleans-Metairie-Kenner,* LA       4781    4488     152      36     105      12'
Line 256: 'New York-Northern New Jersey-Long Island,*'
Line 257: '  NY-NJ-PA                            67207   21424    6900    657

In [14]:

# Inspect the final lines of the 2005 Building Permits file.

print("2005 FILE — FINAL 15 LINES")
print("=" * 65)

for line_number in range(
    max(0, len(lines_2005) - 15),
    len(lines_2005)
):
    print(
        f"Line {line_number + 1}: "
        f"{lines_2005[line_number]!r}"
    )


2005 FILE — FINAL 15 LINES
Line 384: 'Wichita Falls, TX                       565     315      16     164      70       9'
Line 385: 'Wichita, KS                            3349    2962     238     101      48       3'
Line 386: 'Williamsport, PA                        299     261       6       0      32       1'
Line 387: 'Wilmington,* NC                        9206    7685     346      50    1125      77'
Line 388: 'Winchester, VA-WV                      1783    1719       0       0      64       1'
Line 389: 'Winston-Salem, NC                      3994    3690      14      12     278      18'
Line 390: 'Worcester, MA                          3598    3154     102     142     200      11'
Line 391: 'Yakima, WA                              780     739      14       0      27       2'
Line 392: 'York-Hanover, PA                       3116    2870      96      55      95      14'
Line 393: 'Youngstown-Warren-Boardman, OH-PA      1228    1079      60      48      41       7'
Line 394: 'Yu

In [15]:

# Parse 2005 metropolitan records using the six numeric columns.

numeric_pattern = re.compile(
    r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*$"
)

records_2005 = []
pending_name = ""

for line in lines_2005[11:]:

    # Stop at the Census footnote section.
    if line.strip().startswith("* - Metropolitan areas"):
        break

    if not line.strip():
        continue

    match = numeric_pattern.match(line)

    if match:
        metro_name = (
            pending_name + " " + match.group(1).strip()
        ).strip()

        records_2005.append({
            "metro_name": metro_name,
            "year": 2005,
            "permits_total": int(match.group(2)),
            "units_1": int(match.group(3)),
            "units_2": int(match.group(4)),
            "units_3_4": int(match.group(5)),
            "units_5_plus": int(match.group(6)),
            "structures_5_plus": int(match.group(7))
        })

        pending_name = ""

    else:
        pending_name += " " + line.strip()

permits_2005 = pd.DataFrame(records_2005)

print("Total metropolitan records parsed:", len(permits_2005))
print("Unmatched trailing text:", repr(pending_name.strip()))

display(permits_2005.head())
display(permits_2005.tail())


Total metropolitan records parsed: 361
Unmatched trailing text: ''


,metro_name,year,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus
0,"Abilene, TX",2005,346,310,22,0,14,1
1,"Akron, OH",2005,2724,2310,98,105,211,14
2,"Albany, GA",2005,751,689,22,32,8,1
3,"Albany-Schenectady-Troy, NY",2005,3639,2284,10,51,1294,113
4,"Albuquerque,* NM",2005,7038,6712,0,53,273,10


,metro_name,year,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus
356,"Yakima, WA",2005,780,739,14,0,27,2
357,"York-Hanover, PA",2005,3116,2870,96,55,95,14
358,"Youngstown-Warren-Boardman, OH-PA",2005,1228,1079,60,48,41,7
359,"Yuba City, CA",2005,2994,2961,2,0,31,4
360,"Yuma, AZ",2005,2308,2276,2,0,30,6


In [16]:

# Validate housing-unit totals across all 2005 metropolitan records.

permits_2005["calculated_total"] = (
    permits_2005["units_1"]
    + permits_2005["units_2"]
    + permits_2005["units_3_4"]
    + permits_2005["units_5_plus"]
)

permits_2005["total_difference"] = (
    permits_2005["permits_total"]
    - permits_2005["calculated_total"]
)

mismatches_2005 = permits_2005[
    permits_2005["total_difference"] != 0
]

print("Total records:", len(permits_2005))
print("Records with mismatched totals:", len(mismatches_2005))

display(
    mismatches_2005[
        ["metro_name", "permits_total",
         "calculated_total", "total_difference"]
    ]
)


Total records: 361
Records with mismatched totals: 0


,metro_name,permits_total,calculated_total,total_difference


In [17]:

# Verify all seven target metropolitan areas in the 2005 dataset.

target_names_2005 = {
    "Charleston": "Charleston-North Charleston",
    "Dallas": "Dallas-Fort Worth-Arlington",
    "Kansas City": "Kansas City",
    "Las Vegas": "Las Vegas-Paradise",
    "Nashville": "Nashville-Davidson--Murfreesboro",
    "New York": "New York-Northern New Jersey-Long Island",
    "San Francisco": "San Francisco-Oakland-Fremont"
}

selected_records_2005 = []

for label, name in target_names_2005.items():
    matches = permits_2005[
        permits_2005["metro_name"].str.startswith(
            name, na=False
        )
    ]

    print(f"{label}: {len(matches)} match(es)")

    if len(matches) == 1:
        selected_records_2005.append(matches.iloc[0])

selected_2005 = pd.DataFrame(selected_records_2005)

print("\nSelected metropolitan records:", len(selected_2005))

if not selected_2005.empty:
    display(
        selected_2005[
            ["metro_name", "year", "permits_total",
             "calculated_total"]
        ]
    )


Charleston: 1 match(es)
Dallas: 1 match(es)
Kansas City: 1 match(es)
Las Vegas: 1 match(es)
Nashville: 1 match(es)
New York: 1 match(es)
San Francisco: 1 match(es)

Selected metropolitan records: 7


,metro_name,year,permits_total,calculated_total
58,"Charleston-North Charleston,* SC",2005,10818,10818
80,"Dallas-Fort Worth-Arlington,* TX",2005,59895,59895
168,"Kansas City,* MO-KS",2005,15218,15218
184,"Las Vegas-Paradise,* NV",2005,39237,39237
229,"Nashville-Davidson--Murfreesboro,* TN",2005,16654,16654
232,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",2005,67207,67207
290,"San Francisco-Oakland-Fremont,* CA",2005,14883,14883


In [18]:

# Independently count rows containing six numeric fields
# in the 2005 source file.

numeric_rows_2005 = []

for line_number, line in enumerate(lines_2005, start=1):

    # Skip the table header.
    if line_number <= 11:
        continue

    # Stop at the footnote section.
    if line.strip().startswith("* - Metropolitan areas"):
        break

    if numeric_pattern.match(line):
        numeric_rows_2005.append(line_number)

print("Source rows with six numeric fields:", len(numeric_rows_2005))
print("Successfully parsed records:", len(permits_2005))

print(
    "Difference:",
    len(numeric_rows_2005) - len(permits_2005)
)

print(
    "Duplicate metropolitan names:",
    permits_2005["metro_name"].duplicated().sum()
)


Source rows with six numeric fields: 361
Successfully parsed records: 361
Difference: 0
Duplicate metropolitan names: 0


In [19]:

# Inspect the 2010 historical Building Permits file.

url_2010 = (
    "https://www.census.gov/construction/bps/txt/tb3u2010.txt"
)

response_2010 = requests.get(
    url_2010,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response_2010.raise_for_status()

lines_2010 = response_2010.text.splitlines()

print("2010 FILE INSPECTION")
print("=" * 65)
print("Total lines:", len(lines_2010))

print("\nFIRST 15 LINES:\n")

for line in lines_2010[:15]:
    print(repr(line))

print("\nLAST 5 LINES:\n")

for line in lines_2010[-5:]:
    print(repr(line))


2010 FILE INSPECTION
Total lines: 408

FIRST 15 LINES:

'Table 3au. New Privately Owned Housing Units Authorized'
'           Unadjusted Units by Metropolitan Area'
' '
'           Annual 2010'
'                                                                                        Num of'
'                                                                                        Struc-'
'                                                                                        tures'
'                                                                                        With'
'                                                                      3 and 4  5 Units  5 Units'
'CSA CBSA  Name                                   Total 1 Unit 2 Units   Units  or more  or more'
''
'999 10180 Abilene, TX                             390     272       2       0     116       7'
'184 10420 Akron, OH                               736     657       2      27      50       9'
'999 10500 Albany, GA         

In [20]:

# Test the existing CBSA-based parser against 2010.

records_2010 = []
current_record = None

for line in lines_2010:

    # Stop at the Census footnote section.
    if line.strip().startswith(
        "* - Metropolitan areas where all permit offices"
    ):
        break

    match = record_pattern.match(line)

    if match:
        if current_record is not None:
            records_2010.append(current_record)

        current_record = {
            "csa": match.group(1),
            "cbsa": match.group(2),
            "text": match.group(3).strip()
        }

    elif current_record is not None and line.strip():
        current_record["text"] += " " + line.strip()

if current_record is not None:
    records_2010.append(current_record)

parsed_records_2010 = []

for record in records_2010:
    match = re.search(
        r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
        record["text"]
    )

    if match:
        parsed_records_2010.append({
            "csa": record["csa"],
            "cbsa": record["cbsa"],
            "metro_name": match.group(1).strip(),
            "permits_total": int(match.group(2)),
            "units_1": int(match.group(3)),
            "units_2": int(match.group(4)),
            "units_3_4": int(match.group(5)),
            "units_5_plus": int(match.group(6)),
            "structures_5_plus": int(match.group(7)),
            "year": 2010
        })

permits_2010 = pd.DataFrame(parsed_records_2010)

selected_2010 = permits_2010[
    permits_2010["cbsa"].isin(target_cbsas.keys())
].copy()

print("2010 PARSER TEST")
print("=" * 65)
print("Record starts identified:", len(records_2010))
print("Successfully parsed records:", len(permits_2010))
print("Difference:", len(records_2010) - len(permits_2010))
print("Selected MSA records:", len(selected_2010))
print("Unique selected CBSAs:", selected_2010["cbsa"].nunique())

display(
    selected_2010[
        ["cbsa", "metro_name", "year", "permits_total"]
    ]
)


2010 PARSER TEST
Record starts identified: 366
Successfully parsed records: 366
Difference: 0
Selected MSA records: 7
Unique selected CBSAs: 7


,cbsa,metro_name,year,permits_total
59,16700,"Charleston-North Charleston-Summerville,* SC",2010,3060
82,19100,"Dallas-Fort Worth-Arlington,* TX",2010,19558
169,28140,"Kansas City,* MO-KS",2010,2714
186,29820,"Las Vegas-Paradise,* NV",2010,5474
233,34980,"Nashville-Davidson--Murfreesboro-Franklin,* TN",2010,5092
236,35620,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",2010,18668
296,41860,"San Francisco-Oakland-Fremont,* CA",2010,4621


In [21]:

# Validate all 2010 housing-unit totals and CBSA uniqueness.

permits_2010["calculated_total"] = (
    permits_2010["units_1"]
    + permits_2010["units_2"]
    + permits_2010["units_3_4"]
    + permits_2010["units_5_plus"]
)

permits_2010["total_difference"] = (
    permits_2010["permits_total"]
    - permits_2010["calculated_total"]
)

mismatches_2010 = permits_2010[
    permits_2010["total_difference"] != 0
]

duplicate_cbsas_2010 = permits_2010[
    permits_2010.duplicated(subset=["cbsa"], keep=False)
]

missing_targets_2010 = (
    set(target_cbsas.keys()) - set(permits_2010["cbsa"])
)

print("2010 VALIDATION")
print("=" * 65)
print("Total records:", len(permits_2010))
print("Arithmetic mismatches:", len(mismatches_2010))
print("Duplicate CBSA records:", len(duplicate_cbsas_2010))
print("Missing target CBSAs:", sorted(missing_targets_2010))

if not mismatches_2010.empty:
    display(mismatches_2010)

if not duplicate_cbsas_2010.empty:
    display(duplicate_cbsas_2010)


2010 VALIDATION
Total records: 366
Arithmetic mismatches: 0
Duplicate CBSA records: 0
Missing target CBSAs: []


In [22]:

# Compare the standard and new-universe 2014 Census files.

urls_2014 = {
    "Standard 2014": (
        "https://www.census.gov/construction/bps/txt/tb3u2014.txt"
    ),
    "New-universe 2014": (
        "https://www.census.gov/construction/bps/txt/tb3u2014_newuniv.txt"
    )
}

files_2014 = {}

for label, url in urls_2014.items():
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    file_lines = response.text.splitlines()
    files_2014[label] = file_lines

    print(f"\n{label.upper()}")
    print("=" * 65)
    print("Total lines:", len(file_lines))

    print("\nFIRST 12 LINES:")
    for line in file_lines[:12]:
        print(repr(line))

    print("\nLAST 5 LINES:")
    for line in file_lines[-5:]:
        print(repr(line))



STANDARD 2014
Total lines: 419

FIRST 12 LINES:
'Table 3au. New Privately Owned Housing Units Authorized'
'           Unadjusted Units by Metropolitan Area'
' '
'           Annual 2014'
'                                                                                        Num of'
'                                                                                        Struc-'
'                                                                                        tures'
'                                                                                        With'
'                                                                      3 and 4  5 Units  5 Units'
'CSA CBSA  Name                                   Total 1 Unit 2 Units   Units  or more  or more'
''
'999 10180 Abilene, TX                             284     284       0       0       0       0'

LAST 5 LINES:
'472 49700 Yuba City, CA                           280     270      10       0       0       0'
'999 49740 Yuma, AZ   

In [23]:

# Compare the complete contents of both 2014 files.

standard_lines = files_2014["Standard 2014"]
new_universe_lines = files_2014["New-universe 2014"]

differences = []

for line_number, (standard, new) in enumerate(
    zip(standard_lines, new_universe_lines), start=1
):
    if standard != new:
        differences.append({
            "line_number": line_number,
            "standard": standard,
            "new_universe": new
        })

print("2014 FILE COMPARISON")
print("=" * 65)
print("Standard file lines:", len(standard_lines))
print("New-universe file lines:", len(new_universe_lines))
print("Differing lines:", len(differences))
print("Files identical:", standard_lines == new_universe_lines)

print("\nFIRST 10 DIFFERENCES:")

for difference in differences[:10]:
    print(f"\nLine {difference['line_number']}")
    print("Standard:    ", repr(difference["standard"]))
    print("New universe:", repr(difference["new_universe"]))


2014 FILE COMPARISON
Standard file lines: 419
New-universe file lines: 419
Differing lines: 108
Files identical: False

FIRST 10 DIFFERENCES:

Line 13
Standard:     '184 10420 Akron, OH                               763     684       0      16      63       1'
New universe: '184 10420 Akron, OH                               764     685       0      16      63       1'

Line 26
Standard:     '118 11540 Appleton, WI                            930     508      36      40     346      35'
New universe: '118 11540 Appleton, WI                            935     513      36      40     346      35'

Line 28
Standard:     '122 12020 Athens-Clarke County, GA                930     502       2       4     422      19'
New universe: '122 12020 Athens-Clarke County, GA                932     504       2       4     422      19'

Line 33
Standard:     '999 12420 Austin-Round Rock,* TX                19949   11515     256     188    7990     273'
New universe: '999 12420 Austin-Round Rock,* TX     

In [24]:

# Compare 2014 standard and new-universe values
# for our seven target metropolitan areas.

comparison_2014 = []

for label, file_lines in files_2014.items():

    for line in file_lines:

        match = record_pattern.match(line)

        if not match:
            continue

        cbsa = match.group(2)

        if cbsa not in target_cbsas:
            continue

        numeric_match = re.search(
            r"\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*$",
            line
        )

        if numeric_match:
            comparison_2014.append({
                "version": label,
                "cbsa": cbsa,
                "metro": target_cbsas[cbsa],
                "permits_total": int(numeric_match.group(1))
            })

comparison_df_2014 = pd.DataFrame(comparison_2014)

comparison_pivot_2014 = comparison_df_2014.pivot(
    index=["cbsa", "metro"],
    columns="version",
    values="permits_total"
)

comparison_pivot_2014["difference"] = (
    comparison_pivot_2014["New-universe 2014"]
    - comparison_pivot_2014["Standard 2014"]
)

print("2014 TARGET MSA COMPARISON")
print("=" * 65)

display(comparison_pivot_2014.reset_index())

print(
    "\nMSAs with changed totals:",
    (comparison_pivot_2014["difference"] != 0).sum()
)


2014 TARGET MSA COMPARISON


version,cbsa,metro,New-universe 2014,Standard 2014,difference
0,16700,"Charleston–North Charleston, SC",6155,6155,0
1,19100,"Dallas–Fort Worth, TX",43752,41418,2334
2,28140,"Kansas City, MO–KS",8207,8201,6
3,29820,"Las Vegas, NV",10036,10036,0
4,41860,"San Francisco, CA",10001,10001,0



MSAs with changed totals: 2


In [25]:

# Inspect Nashville and New York in both 2014 files.

target_codes = {
    "34980": "Nashville",
    "35620": "New York"
}

for version, file_lines in files_2014.items():

    print(f"\n{version.upper()}")
    print("=" * 65)

    for cbsa, metro in target_codes.items():

        print(f"\n{metro} (CBSA {cbsa}):")

        matching_lines = [
            i for i, line in enumerate(file_lines)
            if record_pattern.match(line)
            and record_pattern.match(line).group(2) == cbsa
        ]

        for i in matching_lines:
            for j in range(i, min(i + 3, len(file_lines))):
                print(f"Line {j + 1}: {file_lines[j]!r}")



STANDARD 2014

Nashville (CBSA 34980):
Line 269: '400 34980 Nashville-Davidson--Murfreesboro--Franklin,'
Line 270: '  TN                                            14944    9075      66      28    5775     163'
Line 271: '404 35100 New Bern, NC                            264     264       0       0       0       0'

New York (CBSA 35620):
Line 274: '408 35620 New York-Newark-Jersey City,*'
Line 275: '  NY-NJ-PA                                      47984   11799    1338    1071   33776    1046'
Line 276: '515 35660 Niles-Benton Harbor, MI                 184     184       0       0       0       0'

NEW-UNIVERSE 2014

Nashville (CBSA 34980):
Line 269: '400 34980 Nashville-Davidson--Murfreesboro--Franklin,'
Line 270: '  TN                                            15040    9171      66      28    5775     163'
Line 271: '404 35100 New Bern, NC                            264     264       0       0       0       0'

New York (CBSA 35620):
Line 274: '408 35620 New York-Newark-Jersey City

In [26]:

# Parse both 2014 versions, including multiline MSA names.

parsed_2014_versions = {}

for version, file_lines in files_2014.items():

    records = []
    current_record = None

    for line in file_lines:

        if line.strip().startswith("* - Metropolitan areas"):
            break

        match = record_pattern.match(line)

        if match:
            if current_record is not None:
                records.append(current_record)

            current_record = {
                "csa": match.group(1),
                "cbsa": match.group(2),
                "text": match.group(3).strip()
            }

        elif current_record is not None and line.strip():
            current_record["text"] += " " + line.strip()

    if current_record is not None:
        records.append(current_record)

    parsed = []
    failed = []

    for record in records:
        match = re.search(
            r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
            record["text"]
        )

        if match:
            parsed.append({
                "cbsa": record["cbsa"],
                "metro_name": match.group(1).strip(),
                "year": 2014,
                "permits_total": int(match.group(2)),
                "units_1": int(match.group(3)),
                "units_2": int(match.group(4)),
                "units_3_4": int(match.group(5)),
                "units_5_plus": int(match.group(6)),
                "structures_5_plus": int(match.group(7))
            })
        else:
            failed.append(record)

    df = pd.DataFrame(parsed)

    calculated = (
        df["units_1"] + df["units_2"]
        + df["units_3_4"] + df["units_5_plus"]
    )

    parsed_2014_versions[version] = df

    print(f"\n{version.upper()}")
    print("=" * 55)
    print("Record starts:", len(records))
    print("Successfully parsed:", len(df))
    print("Failed records:", len(failed))
    print("Arithmetic mismatches:",
          (df["permits_total"] != calculated).sum())
    print("Duplicate CBSAs:", df["cbsa"].duplicated().sum())
    print("Target MSAs found:",
          df["cbsa"].isin(target_cbsas).sum())



STANDARD 2014
Record starts: 381
Successfully parsed: 381
Failed records: 0
Arithmetic mismatches: 0
Duplicate CBSAs: 0
Target MSAs found: 7

NEW-UNIVERSE 2014
Record starts: 381
Successfully parsed: 381
Failed records: 0
Arithmetic mismatches: 0
Duplicate CBSAs: 0
Target MSAs found: 7


In [27]:

# Verify that our historical Building Permits
# notebook environment is ready.

datasets = {
    "2005": permits_2005,
    "2010": permits_2010,
    "2018": permits_2018,
    "2014 Standard": parsed_2014_versions["Standard 2014"],
    "2014 New Universe": parsed_2014_versions["New-universe 2014"]
}

print("HISTORICAL BUILDING PERMITS — SESSION CHECK")
print("=" * 55)

for label, df in datasets.items():
    print(f"{label}: {len(df)} records")

print("\nTarget CBSAs configured:", len(target_cbsas))
print("2014 source versions:", list(files_2014.keys()))


HISTORICAL BUILDING PERMITS — SESSION CHECK
2005: 361 records
2010: 366 records
2018: 382 records
2014 Standard: 381 records
2014 New Universe: 381 records

Target CBSAs configured: 7
2014 source versions: ['Standard 2014', 'New-universe 2014']


In [28]:

# Inspect the 2006 historical Building Permits file.

url_2006 = (
    "https://www.census.gov/construction/bps/txt/tb3u2006.txt"
)

response_2006 = requests.get(
    url_2006,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

response_2006.raise_for_status()

lines_2006 = response_2006.text.splitlines()

print("2006 FILE INSPECTION")
print("=" * 65)
print("Total lines:", len(lines_2006))

print("\nFIRST 15 LINES:\n")

for line in lines_2006[:15]:
    print(repr(line))

print("\nLAST 5 LINES:\n")

for line in lines_2006[-5:]:
    print(repr(line))


2006 FILE INSPECTION
Total lines: 398

FIRST 15 LINES:

'Table 3au. New Privately Owned Housing Units Authorized'
'           Unadjusted Units by Metropolitan Area'
' '
'           Annual 2006'
'                                                                              Num of'
'                                                                              Struc-'
'                                                                              tures'
'                                                                              With'
'                                                          3 and 4  5 Units  5 Units'
'                                     Total 1 Unit 2 Units   Units  or more  or more'
''
'Abilene, TX                             365     333      16       0      16       1'
'Akron, OH                              2209    1746      24     100     339      18'
'Albany, GA                              727     647      40      32       8       1'
'Albany-Schenectady-Troy, NY

In [29]:

# Extract only our seven target MSAs from the 2006 Census file.

target_prefixes_2006 = {
    "16700": "Charleston-",
    "19100": "Dallas-Fort Worth-",
    "28140": "Kansas City,",
    "29820": "Las Vegas-",
    "34980": "Nashville-",
    "35620": "New York-",
    "41860": "San Francisco-"
}

records_2006 = []
pending_name = ""

for line in lines_2006[11:]:
    if line.strip().startswith("* - Metropolitan areas"):
        break

    if not line.strip():
        continue

    match = numeric_pattern.match(line)

    if match:
        metro_name = (
            pending_name + " " + match.group(1).strip()
        ).strip()

        for cbsa, prefix in target_prefixes_2006.items():
            if metro_name.startswith(prefix):
                records_2006.append({
                    "year": 2006,
                    "cbsa": cbsa,
                    "metro_name": metro_name,
                    "permits_total": int(match.group(2)),
                    "units_1": int(match.group(3)),
                    "units_2": int(match.group(4)),
                    "units_3_4": int(match.group(5)),
                    "units_5_plus": int(match.group(6)),
                    "structures_5_plus": int(match.group(7))
                })
                break

        pending_name = ""
    else:
        pending_name += " " + line.strip()

permits_2006 = pd.DataFrame(records_2006)

print("2006 TARGET MSA EXTRACTION")
print("=" * 60)

print("MSAs found:", len(permits_2006))
print("Unique CBSAs:", permits_2006["cbsa"].nunique())

permits_2006["components_sum"] = (
    permits_2006["units_1"]
    + permits_2006["units_2"]
    + permits_2006["units_3_4"]
    + permits_2006["units_5_plus"]
)

print(
    "Arithmetic mismatches:",
    (
        permits_2006["permits_total"]
        != permits_2006["components_sum"]
    ).sum()
)

display(
    permits_2006[
        ["year", "cbsa", "metro_name", "permits_total"]
    ].sort_values("cbsa")
)


2006 TARGET MSA EXTRACTION
MSAs found: 7
Unique CBSAs: 7
Arithmetic mismatches: 0


,year,cbsa,metro_name,permits_total
0,2006,16700,"Charleston-North Charleston,* SC",9073
1,2006,19100,"Dallas-Fort Worth-Arlington,* TX",56514
2,2006,28140,"Kansas City,* MO-KS",13205
3,2006,29820,"Las Vegas-Paradise,* NV",33728
4,2006,34980,"Nashville-Davidson--Murfreesboro,* TN",15327
5,2006,35620,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",60987
6,2006,41860,"San Francisco-Oakland-Fremont,* CA",14483


In [30]:

# Inspect historical Census file formats for 2007–2009.
# This does not parse or validate all metropolitan records.

years_to_inspect = [2007, 2008, 2009]

for year in years_to_inspect:
    url = (
        f"https://www.census.gov/construction/bps/"
        f"txt/tb3u{year}.txt"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    lines = response.text.splitlines()

    has_cbsa_header = any(
        "CBSA" in line.upper()
        for line in lines[:15]
    )

    print(f"\nYEAR: {year}")
    print("-" * 55)
    print("Total lines:", len(lines))
    print(
        "Format:",
        "CBSA-coded" if has_cbsa_header
        else "Name-based (no CBSA header)"
    )

    print("First metropolitan record:")
    for line in lines[10:20]:
        if line.strip():
            print(repr(line))
            break

    print("Final nonblank lines:")
    for line in [x for x in lines[-5:] if x.strip()]:
        print(repr(line))



YEAR: 2007
-------------------------------------------------------
Total lines: 400
Format: Name-based (no CBSA header)
First metropolitan record:
'Abilene, TX                             293     283      10       0       0       0'
Final nonblank lines:
'Yuba City, CA                          1028     928       0       4      96      11'
'Yuma, AZ                               1044     967       8       0      69       4'
'* - Metropolitan areas where all permit offices are '
'    requested to report monthly.'

YEAR: 2008
-------------------------------------------------------
Total lines: 402
Format: Name-based (no CBSA header)
First metropolitan record:
'Abilene, TX                             196     190       6       0       0       0'
Final nonblank lines:
'Yuba City, CA                           261     199       2       0      60       7'
'Yuma, AZ                                827     786       8       0      33       5'
'* - Metropolitan areas where all permit offices are '

In [31]:

# Extract seven target MSAs for 2007–2009.
# Uses the previously validated parsing patterns.

new_years = []

for year in [2007, 2008, 2009]:

    url = (
        f"https://www.census.gov/construction/bps/"
        f"txt/tb3u{year}.txt"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()
    lines = response.text.splitlines()

    rows = []

    if year < 2009:
        # Name-based format, including multiline names.
        pending_name = ""

        for line in lines[11:]:
            if line.strip().startswith("* - Metropolitan areas"):
                break
            if not line.strip():
                continue

            match = numeric_pattern.match(line)

            if match:
                name = (
                    pending_name + " " + match.group(1).strip()
                ).strip()

                for cbsa, prefix in target_prefixes_2006.items():
                    if name.startswith(prefix):
                        rows.append({
                            "year": year,
                            "cbsa": cbsa,
                            "metro_name": name,
                            "permits_total": int(match.group(2)),
                            "units_1": int(match.group(3)),
                            "units_2": int(match.group(4)),
                            "units_3_4": int(match.group(5)),
                            "units_5_plus": int(match.group(6)),
                            "structures_5_plus": int(match.group(7))
                        })
                        break

                pending_name = ""

            else:
                pending_name += " " + line.strip()

    else:
        # CBSA-coded format, including multiline names.
        records = []
        current = None

        for line in lines:
            if line.strip().startswith("* - Metropolitan areas"):
                break

            match = record_pattern.match(line)

            if match:
                if current is not None:
                    records.append(current)

                current = {
                    "cbsa": match.group(2),
                    "text": match.group(3).strip()
                }

            elif current is not None and line.strip():
                current["text"] += " " + line.strip()

        if current is not None:
            records.append(current)

        for record in records:
            if record["cbsa"] not in target_cbsas:
                continue

            match = re.search(
                r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
                record["text"]
            )

            if match:
                rows.append({
                    "year": year,
                    "cbsa": record["cbsa"],
                    "metro_name": match.group(1).strip(),
                    "permits_total": int(match.group(2)),
                    "units_1": int(match.group(3)),
                    "units_2": int(match.group(4)),
                    "units_3_4": int(match.group(5)),
                    "units_5_plus": int(match.group(6)),
                    "structures_5_plus": int(match.group(7))
                })

    df = pd.DataFrame(rows)

    assert len(df) == 7, f"{year}: expected 7 rows, got {len(df)}"
    assert df["cbsa"].nunique() == 7, f"{year}: duplicate or missing CBSA"

    components = (
        df["units_1"] + df["units_2"]
        + df["units_3_4"] + df["units_5_plus"]
    )

    assert (df["permits_total"] == components).all(), (
        f"{year}: arithmetic mismatch"
    )

    new_years.append(df)

    print(
        f"{year}: 7 MSAs, 7 unique CBSAs, "
        "0 arithmetic mismatches"
    )

permits_2007_2009 = pd.concat(
    new_years, ignore_index=True
)

print("\nTARGET HOUSING UNITS AUTHORIZED")
display(
    permits_2007_2009.pivot(
        index="cbsa",
        columns="year",
        values="permits_total"
    ).rename(index=target_cbsas)
)


2007: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2008: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2009: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches

TARGET HOUSING UNITS AUTHORIZED


year,2007,2008,2009
cbsa,,,
"Charleston–North Charleston, SC",6780,5172,2949
"Dallas–Fort Worth, TX",43568,36321,20370
"Kansas City, MO–KS",8129,5300,3406
"Las Vegas, NV",24089,12537,5688
"Nashville, TN",13567,8188,4933
"New York, NY",56405,51590,16707
"San Francisco, CA",10560,7555,3550


In [32]:

# Extract seven target MSAs for 2011–2013.

years_to_process = [2011, 2012, 2013]
extracted_years = []

for year in years_to_process:
    url = (
        f"https://www.census.gov/construction/bps/"
        f"txt/tb3u{year}.txt"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    lines = response.text.splitlines()

    records = []
    current = None

    for line in lines:
        if line.strip().startswith("* - Metropolitan areas"):
            break

        match = record_pattern.match(line)

        if match:
            if current is not None:
                records.append(current)

            current = {
                "cbsa": match.group(2),
                "text": match.group(3).strip()
            }

        elif current is not None and line.strip():
            current["text"] += " " + line.strip()

    if current is not None:
        records.append(current)

    rows = []

    for record in records:
        if record["cbsa"] not in target_cbsas:
            continue

        match = re.search(
            r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+"
            r"(\d+)\s+(\d+)\s+(\d+)$",
            record["text"]
        )

        if match:
            rows.append({
                "year": year,
                "cbsa": record["cbsa"],
                "metro_name": match.group(1).strip(),
                "permits_total": int(match.group(2)),
                "units_1": int(match.group(3)),
                "units_2": int(match.group(4)),
                "units_3_4": int(match.group(5)),
                "units_5_plus": int(match.group(6)),
                "structures_5_plus": int(match.group(7))
            })

    df = pd.DataFrame(rows)

    assert len(df) == 7, (
        f"{year}: Expected 7 MSAs, found {len(df)}"
    )

    assert df["cbsa"].nunique() == 7, (
        f"{year}: Duplicate or missing CBSAs"
    )

    component_sum = (
        df["units_1"]
        + df["units_2"]
        + df["units_3_4"]
        + df["units_5_plus"]
    )

    assert (df["permits_total"] == component_sum).all(), (
        f"{year}: Housing-unit arithmetic mismatch"
    )

    extracted_years.append(df)

    print(
        f"{year}: 7 MSAs, 7 unique CBSAs, "
        "0 arithmetic mismatches"
    )

permits_2011_2013 = pd.concat(
    extracted_years,
    ignore_index=True
)

print("\nHOUSING UNITS AUTHORIZED — 2011–2013")

display(
    permits_2011_2013.pivot(
        index="cbsa",
        columns="year",
        values="permits_total"
    ).rename(index=target_cbsas)
)


2011: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2012: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2013: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches

HOUSING UNITS AUTHORIZED — 2011–2013


year,2011,2012,2013
cbsa,,,
"Charleston–North Charleston, SC",3822,4593,5417
"Dallas–Fort Worth, TX",24827,35042,37910
"Kansas City, MO–KS",3287,4981,7532
"Las Vegas, NV",5147,7375,8573
"Nashville, TN",5394,8247,10889
"New York, NY",21539,26912,39824
"San Francisco, CA",5783,9163,10922


In [33]:

# Reusable extraction function for CBSA-coded Census text files.
# Extracts and validates only the seven target metropolitan areas.

def extract_target_permits(year):
    url = (
        f"https://www.census.gov/construction/bps/"
        f"txt/tb3u{year}.txt"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    lines = response.text.splitlines()
    records = []
    current = None

    for line in lines:
        if line.strip().startswith("* - Metropolitan areas"):
            break

        match = record_pattern.match(line)

        if match:
            if current is not None:
                records.append(current)

            current = {
                "cbsa": match.group(2),
                "text": match.group(3).strip()
            }

        elif current is not None and line.strip():
            current["text"] += " " + line.strip()

    if current is not None:
        records.append(current)

    rows = []
    failed_targets = []

    for record in records:
        if record["cbsa"] not in target_cbsas:
            continue

        match = re.search(
            r"^(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+"
            r"(\d+)\s+(\d+)\s+(\d+)$",
            record["text"]
        )

        if not match:
            failed_targets.append(record["cbsa"])
            continue

        rows.append({
            "year": year,
            "cbsa": record["cbsa"],
            "metro_name": match.group(1).strip(),
            "permits_total": int(match.group(2)),
            "units_1": int(match.group(3)),
            "units_2": int(match.group(4)),
            "units_3_4": int(match.group(5)),
            "units_5_plus": int(match.group(6)),
            "structures_5_plus": int(match.group(7)),
            "source_url": url
        })

    df = pd.DataFrame(rows)

    assert not failed_targets, (
        f"{year}: Failed target records: {failed_targets}"
    )

    assert len(df) == 7, (
        f"{year}: Expected 7 rows, found {len(df)}"
    )

    assert set(df["cbsa"]) == set(target_cbsas), (
        f"{year}: Missing or unexpected CBSAs"
    )

    assert df["cbsa"].is_unique, (
        f"{year}: Duplicate CBSAs"
    )

    components = (
        df["units_1"]
        + df["units_2"]
        + df["units_3_4"]
        + df["units_5_plus"]
    )

    assert (df["permits_total"] == components).all(), (
        f"{year}: Arithmetic mismatch"
    )

    print(
        f"{year}: 7 MSAs, 7 unique CBSAs, "
        "0 arithmetic mismatches"
    )

    return df


# Process 2015–2017.
permits_2015_2017 = pd.concat(
    [extract_target_permits(year) for year in (2015, 2016, 2017)],
    ignore_index=True
)

print("\nHOUSING UNITS AUTHORIZED — 2015–2017")

display(
    permits_2015_2017.pivot(
        index="cbsa",
        columns="year",
        values="permits_total"
    ).rename(index=target_cbsas)
)


2015: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2016: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2017: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches

HOUSING UNITS AUTHORIZED — 2015–2017


year,2015,2016,2017
cbsa,,,
"Charleston–North Charleston, SC",6590,6974,7267
"Dallas–Fort Worth, TX",57146,55800,62524
"Kansas City, MO–KS",8954,10389,9851
"Las Vegas, NV",10605,13577,14073
"Nashville, TN",18291,20182,20631
"New York, NY",86424,43231,50578
"San Francisco, CA",13386,14787,16952


In [34]:

# Inspect the 2019 Census Building Permits Excel workbook.
# No extraction or transformation yet.

import io

url_2019 = (
    "https://www.census.gov/construction/bps/"
    "xls/msaannual_201999.xls"
)

response_2019 = requests.get(
    url_2019,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response_2019.raise_for_status()

print("2019 WORKBOOK INSPECTION")
print("=" * 60)
print("HTTP status:", response_2019.status_code)
print("File size (bytes):", len(response_2019.content))

workbook_2019 = pd.ExcelFile(
    io.BytesIO(response_2019.content)
)

print("Sheet names:", workbook_2019.sheet_names)

for sheet in workbook_2019.sheet_names:
    preview = pd.read_excel(
        workbook_2019,
        sheet_name=sheet,
        header=None,
        nrows=15
    )

    print(f"\nSHEET: {sheet}")
    print("-" * 60)
    display(preview)


2019 WORKBOOK INSPECTION
HTTP status: 200
File size (bytes): 135168
Sheet names: ['MSA Units', 'MSA Value']

SHEET: MSA Units
------------------------------------------------------------


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Unadjusted Units for Regions, Divisions, and States",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Annual 2019,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",370,354,16,0,0,0
8,184,10420,"Akron, OH",945,856,2,32,55,1
9,999,10500,"Albany, GA",402,182,0,0,220,10



SHEET: MSA Value
------------------------------------------------------------


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Valuation by Metropolitan Area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Annual 2019,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",74917,72824,2093,0,0,0
8,184,10420,"Akron, OH",347509,343432,300,3040,737,1
9,999,10500,"Albany, GA",57483,24644,0,0,32839,10


In [35]:

# Extract and validate the seven target MSAs from 2019.

df_2019 = pd.read_excel(
    workbook_2019,
    sheet_name="MSA Units",
    header=5
)

# Standardize CBSA identifiers as five-character strings.
df_2019["CBSA"] = (
    pd.to_numeric(df_2019["CBSA"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

# Keep only the seven target metropolitan areas.
permits_2019 = df_2019[
    df_2019["CBSA"].isin(target_cbsas)
].copy()

# Standardize column names.
permits_2019 = permits_2019.rename(columns={
    "CBSA": "cbsa",
    "Name": "metro_name",
    "Total": "permits_total",
    "1 Unit": "units_1",
    "2 Units": "units_2",
    "3 and 4 Units": "units_3_4",
    "5 Units or More": "units_5_plus",
    "Num of Structures With 5 Units or More": "structures_5_plus"
})

permits_2019["year"] = 2019

numeric_columns = [
    "permits_total", "units_1", "units_2",
    "units_3_4", "units_5_plus", "structures_5_plus"
]

for column in numeric_columns:
    permits_2019[column] = pd.to_numeric(
        permits_2019[column], errors="raise"
    )

# Validate target coverage and arithmetic.
assert len(permits_2019) == 7, (
    f"Expected 7 MSAs, found {len(permits_2019)}"
)

assert set(permits_2019["cbsa"]) == set(target_cbsas), (
    "Missing or unexpected CBSA identifiers"
)

assert permits_2019["cbsa"].is_unique, (
    "Duplicate CBSA identifiers"
)

components = (
    permits_2019["units_1"]
    + permits_2019["units_2"]
    + permits_2019["units_3_4"]
    + permits_2019["units_5_plus"]
)

assert (
    permits_2019["permits_total"] == components
).all(), "Housing-unit arithmetic mismatch"

print("2019 TARGET MSA VALIDATION")
print("=" * 55)
print("MSAs found:", len(permits_2019))
print("Unique CBSAs:", permits_2019["cbsa"].nunique())
print(
    "Arithmetic mismatches:",
    (permits_2019["permits_total"] != components).sum()
)

display(
    permits_2019[
        ["year", "cbsa", "metro_name", "permits_total"]
    ].sort_values("cbsa")
)


2019 TARGET MSA VALIDATION
MSAs found: 7
Unique CBSAs: 7
Arithmetic mismatches: 0


,year,cbsa,metro_name,permits_total
64,2019,16700,"Charleston-North Charleston, SC",6695.0
87,2019,19100,"Dallas-Fort Worth-Arlington, TX",62708.0
181,2019,28140,"Kansas City, MO-KS",9347.0
198,2019,29820,"Las Vegas-Henderson-Paradise, NV",13903.0
246,2019,34980,"Nashville-Davidson--Murfreesboro--Franklin, TN",22702.0
250,2019,35620,"New York-Newark-Jersey City, NY-NJ-PA",61168.0
308,2019,41860,"San Francisco-Oakland-Berkeley, CA",13881.0


In [36]:

# Extract and validate 2020–2021 Census Building Permits.
# Retain only the seven target metropolitan areas.

years_to_process = [2020, 2021]
extracted_years = []

column_map = {
    "CBSA": "cbsa",
    "Name": "metro_name",
    "Total": "permits_total",
    "1 Unit": "units_1",
    "2 Units": "units_2",
    "3 and 4 Units": "units_3_4",
    "5 Units or More": "units_5_plus",
    "Num of Structures With 5 Units or More": "structures_5_plus"
}

numeric_columns = [
    "permits_total", "units_1", "units_2",
    "units_3_4", "units_5_plus", "structures_5_plus"
]

for year in years_to_process:
    url = (
        "https://www.census.gov/construction/bps/"
        f"xls/msaannual_{year}99.xls"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    workbook = pd.ExcelFile(io.BytesIO(response.content))

    assert "MSA Units" in workbook.sheet_names, (
        f"{year}: MSA Units worksheet not found"
    )

    df = pd.read_excel(
        workbook,
        sheet_name="MSA Units",
        header=5
    )

    df["CBSA"] = (
        pd.to_numeric(df["CBSA"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    permits = df[
        df["CBSA"].isin(target_cbsas)
    ].copy().rename(columns=column_map)

    permits["year"] = year
    permits["source_url"] = url

    assert len(permits) == 7, (
        f"{year}: Expected 7 MSAs, found {len(permits)}"
    )

    assert set(permits["cbsa"]) == set(target_cbsas), (
        f"{year}: Missing or unexpected CBSAs"
    )

    assert permits["cbsa"].is_unique, (
        f"{year}: Duplicate CBSAs"
    )

    for column in numeric_columns:
        permits[column] = pd.to_numeric(
            permits[column], errors="raise"
        )

    assert permits[numeric_columns].notna().all().all(), (
        f"{year}: Missing housing-unit values"
    )

    assert (permits[numeric_columns] >= 0).all().all(), (
        f"{year}: Negative housing-unit values"
    )

    assert (
        permits[numeric_columns] % 1 == 0
    ).all().all(), (
        f"{year}: Non-integer housing-unit values"
    )

    permits[numeric_columns] = (
        permits[numeric_columns].astype("int64")
    )

    components = (
        permits["units_1"]
        + permits["units_2"]
        + permits["units_3_4"]
        + permits["units_5_plus"]
    )

    assert (
        permits["permits_total"] == components
    ).all(), (
        f"{year}: Arithmetic mismatch"
    )

    extracted_years.append(permits)

    print(
        f"{year}: 7 MSAs, 7 unique CBSAs, "
        "0 arithmetic mismatches"
    )

permits_2020_2021 = pd.concat(
    extracted_years,
    ignore_index=True
)

print("\nHOUSING UNITS AUTHORIZED — 2020–2021")

display(
    permits_2020_2021.pivot(
        index="cbsa",
        columns="year",
        values="permits_total"
    ).rename(index=target_cbsas)
)


2020: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2021: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches

HOUSING UNITS AUTHORIZED — 2020–2021


year,2020,2021
cbsa,,
"Charleston–North Charleston, SC",8604,8282
"Dallas–Fort Worth, TX",60812,78705
"Kansas City, MO–KS",10728,11254
"Las Vegas, NV",14100,16307
"Nashville, TN",27158,32191
"New York, NY",54835,56661
"San Francisco, CA",10156,13606


In [37]:

# Discover the available Census Building Permits files
# for 2022, 2023, and 2024.

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

page_url = (
    "https://www.census.gov/construction/bps/"
    "msaannual.html"
)

response = requests.get(
    page_url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("CENSUS ANNUAL METROPOLITAN PERMITS — SOURCE DISCOVERY")
print("=" * 65)

for year in [2022, 2023, 2024]:
    matches = []

    for anchor in soup.find_all("a", href=True):
        href = anchor["href"]
        label = anchor.get_text(" ", strip=True)

        if str(year) in href or str(year) in label:
            matches.append((
                label,
                urljoin(page_url, href)
            ))

    print(f"\nYEAR: {year}")
    print("-" * 55)

    if matches:
        for label, url in matches:
            print(f"Label: {label}")
            print(f"URL:   {url}")
    else:
        print("No matching links found on the annual MSA page.")


CENSUS ANNUAL METROPOLITAN PERMITS — SOURCE DISCOVERY

YEAR: 2022
-------------------------------------------------------
No matching links found on the annual MSA page.

YEAR: 2023
-------------------------------------------------------
Label: 2023 Methodology Change FAQs
URL:   https://www.census.gov/construction/pdf/bps_2023_faqs.pdf

YEAR: 2024
-------------------------------------------------------
No matching links found on the annual MSA page.


In [38]:

# Test possible Census Excel workbook URLs for 2022–2024.
# Check availability and file signatures without parsing.

import requests

print("CENSUS BUILDING PERMITS — 2022–2024 FILE CHECK")
print("=" * 65)

for year in [2022, 2023, 2024]:
    url = (
        "https://www.census.gov/construction/bps/"
        f"xls/msaannual_{year}99.xls"
    )

    try:
        response = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        signature = response.content[:8]
        is_excel = (
            signature.startswith(b"\xd0\xcf\x11\xe0")
            or signature.startswith(b"PK\x03\x04")
        )

        print(f"\nYEAR: {year}")
        print("HTTP status:", response.status_code)
        print("File size:", len(response.content), "bytes")
        print("Excel signature detected:", is_excel)
        print("URL:", url)

    except requests.RequestException as exc:
        print(f"\nYEAR: {year}")
        print("Request failed:", exc)


CENSUS BUILDING PERMITS — 2022–2024 FILE CHECK

YEAR: 2022
HTTP status: 200
File size: 171520 bytes
Excel signature detected: True
URL: https://www.census.gov/construction/bps/xls/msaannual_202299.xls

YEAR: 2023
HTTP status: 200
File size: 163840 bytes
Excel signature detected: True
URL: https://www.census.gov/construction/bps/xls/msaannual_202399.xls

YEAR: 2024
HTTP status: 404
File size: 15070 bytes
Excel signature detected: False
URL: https://www.census.gov/construction/bps/xls/msaannual_202499.xls


In [39]:

# Inspect 2022 and 2023 Census Building Permits workbooks.

import io

for year in [2022, 2023]:
    url = (
        "https://www.census.gov/construction/bps/"
        f"xls/msaannual_{year}99.xls"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    workbook = pd.ExcelFile(io.BytesIO(response.content))

    print(f"\n{'=' * 65}")
    print(f"YEAR: {year}")
    print("Sheet names:", workbook.sheet_names)

    for sheet in workbook.sheet_names:
        print(f"\nSHEET: {sheet}")

        preview = pd.read_excel(
            workbook,
            sheet_name=sheet,
            header=None,
            nrows=10
        )

        display(preview)



YEAR: 2022
Sheet names: ['MSA Units Ann', 'MSA Value Ann']

SHEET: MSA Units Ann


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Unadjusted Units for Regions, Divisions, and States",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,202299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",799,473,52,0,274,36
8,184,10420,"Akron, OH",3047,796,68,437,1746,137
9,999,10500,"Albany, GA",202,169,0,0,33,2



SHEET: MSA Value Ann


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Valuation by CBSA (in thousands of dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,202299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",143757,114059,5698,0,24000,36
8,184,10420,"Akron, OH",569590,263252,8000,35008,263330,137
9,999,10500,"Albany, GA",28761,25104,0,0,3657,2



YEAR: 2023
Sheet names: ['MSA Units Ann', 'MSA Value Ann']

SHEET: MSA Units Ann


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Unadjusted Units for Regions, Divisions, and States",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,202399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",462,342,56,0,64,9
8,184,10420,"Akron, OH",940,709,4,103,124,18
9,999,10500,"Albany, GA",183,162,2,4,15,3



SHEET: MSA Value Ann


,0,1,2,3,4,5,6,7,8
0,New Privately Owned Housing Units Authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Valuation by CBSA (in thousands of dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,202399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CSA,CBSA,Name,Total,1 Unit,2 Units,3 and 4 Units,5 Units or More,Num of Structures With 5 Units or More
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,999,10180,"Abilene, TX",107662,92755,6407,0,8500,9
8,184,10420,"Akron, OH",280334,240822,803,17758,20951,18
9,999,10500,"Albany, GA",29020,27117,21,220,1662,3


In [40]:

# Extract seven target MSAs from the 2022 and 2023
# Census Building Permits annual Excel workbooks.

extracted_years = []

for year in [2022, 2023]:
    url = (
        "https://www.census.gov/construction/bps/"
        f"xls/msaannual_{year}99.xls"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    df = pd.read_excel(
        io.BytesIO(response.content),
        sheet_name="MSA Units Ann",
        header=5
    )

    df["CBSA"] = (
        pd.to_numeric(df["CBSA"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    permits = (
        df[df["CBSA"].isin(target_cbsas)]
        .copy()
        .rename(columns=column_map)
    )

    permits["year"] = year
    permits["source_url"] = url

    assert len(permits) == 7, (
        f"{year}: Expected 7 MSAs, found {len(permits)}"
    )

    assert set(permits["cbsa"]) == set(target_cbsas), (
        f"{year}: Missing or unexpected CBSAs"
    )

    assert permits["cbsa"].is_unique, (
        f"{year}: Duplicate CBSAs"
    )

    for column in numeric_columns:
        permits[column] = pd.to_numeric(
            permits[column],
            errors="raise"
        )

    assert permits[numeric_columns].notna().all().all(), (
        f"{year}: Missing housing-unit values"
    )

    assert (permits[numeric_columns] >= 0).all().all(), (
        f"{year}: Negative housing-unit values"
    )

    assert (
        permits[numeric_columns] % 1 == 0
    ).all().all(), (
        f"{year}: Non-integer housing-unit values"
    )

    permits[numeric_columns] = (
        permits[numeric_columns].astype("int64")
    )

    components = (
        permits["units_1"]
        + permits["units_2"]
        + permits["units_3_4"]
        + permits["units_5_plus"]
    )

    assert (
        permits["permits_total"] == components
    ).all(), (
        f"{year}: Arithmetic mismatch"
    )

    extracted_years.append(permits)

    print(
        f"{year}: 7 MSAs, 7 unique CBSAs, "
        "0 arithmetic mismatches"
    )

permits_2022_2023 = pd.concat(
    extracted_years,
    ignore_index=True
)

print("\nHOUSING UNITS AUTHORIZED — 2022–2023")

display(
    permits_2022_2023.pivot(
        index="cbsa",
        columns="year",
        values="permits_total"
    ).rename(index=target_cbsas)
)


2022: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches
2023: 7 MSAs, 7 unique CBSAs, 0 arithmetic mismatches

HOUSING UNITS AUTHORIZED — 2022–2023


year,2022,2023
cbsa,,
"Charleston–North Charleston, SC",9491,8573
"Dallas–Fort Worth, TX",77927,68029
"Kansas City, MO–KS",11213,7514
"Las Vegas, NV",13066,13073
"Nashville, TN",28314,23558
"New York, NY",69753,63030
"San Francisco, CA",11180,7530


In [41]:

# Search official Census Building Permits pages for
# references to 2024 metropolitan annual data.

from bs4 import BeautifulSoup
from urllib.parse import urljoin

pages_to_check = {
    "Annual MSA": (
        "https://www.census.gov/construction/bps/msaannual.html"
    ),
    "Historical Metro Units": (
        "https://www.census.gov/construction/bps/"
        "historical/metro_units.html"
    ),
    "Current Data": (
        "https://www.census.gov/construction/bps/current.html"
    )
}

for page_name, page_url in pages_to_check.items():
    print(f"\n{'=' * 65}")
    print(page_name)
    print("=" * 65)

    try:
        response = requests.get(
            page_url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        matches = []

        for anchor in soup.find_all("a", href=True):
            label = anchor.get_text(" ", strip=True)
            href = anchor["href"]
            combined = (label + " " + href).lower()

            if "2024" in combined or "202499" in combined:
                matches.append((
                    label,
                    urljoin(page_url, href)
                ))

        if matches:
            for label, url in matches:
                print("Label:", label)
                print("URL:  ", url)
                print()
        else:
            print("No direct 2024 links found.")

    except requests.RequestException as exc:
        print("Page request failed:", exc)



Annual MSA
No direct 2024 links found.

Historical Metro Units
No direct 2024 links found.

Current Data
No direct 2024 links found.


In [42]:

# Inspect the official 2024 annual CBSA Building Permits file.

url_2024 = (
    "https://www2.census.gov/econ/bps/"
    "CBSA%20%28beginning%20Jan%202024%29/"
    "cbsa2024a.txt"
)

response_2024 = requests.get(url_2024, timeout=30)
response_2024.raise_for_status()

lines_2024 = response_2024.text.splitlines()

print("2024 ANNUAL CBSA FILE INSPECTION")
print("=" * 65)
print("HTTP status:", response_2024.status_code)
print("Total lines:", len(lines_2024))

print("\nFIRST 10 LINES:")
for line in lines_2024[:10]:
    print(repr(line))

print("\nLAST 5 LINES:")
for line in lines_2024[-5:]:
    print(repr(line))

print("\nTARGET CBSA RECORDS:")
for line in lines_2024:
    fields = [field.strip().strip('"') for field in line.split(",")]
    if any(cbsa in fields for cbsa in target_cbsas):
        print(repr(line))


2024 ANNUAL CBSA FILE INSPECTION
HTTP status: 200
Total lines: 924

FIRST 10 LINES:
'Survey,CSA,CBSA,HHEADER,CBSA,,1-unit,,,2-units,,,3-4 units,,,5+ units,,,1-unit rep,,,2-units rep,,,3-4 units rep,,,5+ units rep'
'Date,Code,Code,,Name,Bldgs,Units,Value,Bldgs,Units,Value,Bldgs,Units,Value,Bldgs,Units,Value,Bldgs,Units,Value,Bldgs,Units,Value,      Bldgs,Units,Value,Bldgs,Units,Value'
' '
'202499,101,10180,2,Abilene  TX ,431,431,120144,48,96,14788,0,0,0,0,0,0,431,431,120144,48,96,14788,0,0,0,0,0,0'
'202499,101,45020,5,Sweetwater  TX ,19,19,3663,0,0,0,0,0,0,0,0,0,19,19,3663,0,0,0,0,0,0,0,0,0'
'202499,104,10580,2,Albany-Schenectady-Troy  NY ,1201,1201,426445,17,34,7367,17,63,10029,72,1068,475593,913,913,323928,13,26,4937,13,51,7693,55,979,463685'
'202499,104,11220,5,Amsterdam  NY ,44,44,10379,0,0,0,0,0,0,14,168,36000,31,31,6570,0,0,0,0,0,0,14,168,36000'
'202499,104,24020,2,Glens Falls  NY ,243,243,77733,5,10,1685,0,0,0,7,102,13168,236,236,76062,5,10,1685,0,0,0,7,102,13168'
'202499,104,241

In [43]:

# Extract and validate 2024 annual housing units authorized.
# The 2024 format separates buildings, units, and valuation.

import pandas as pd
import io

df_2024 = pd.read_csv(
    io.StringIO(response_2024.text),
    header=None,
    skiprows=3,
    dtype={2: str}
)

# Select only our seven target CBSAs.
df_2024[2] = df_2024[2].str.strip().str.zfill(5)

target_rows = df_2024[
    df_2024[2].isin(target_cbsas)
].copy()

permits_2024 = pd.DataFrame({
    "year": 2024,
    "cbsa": target_rows[2],
    "metro_name": target_rows[4].str.strip(),
    "metro_type": target_rows[3],
    "units_1": target_rows[6],
    "units_2": target_rows[9],
    "units_3_4": target_rows[12],
    "units_5_plus": target_rows[15],
    "structures_5_plus": target_rows[14]
})

numeric_columns_2024 = [
    "units_1", "units_2", "units_3_4",
    "units_5_plus", "structures_5_plus"
]

for column in numeric_columns_2024:
    permits_2024[column] = pd.to_numeric(
        permits_2024[column],
        errors="raise"
    )

permits_2024["permits_total"] = (
    permits_2024["units_1"]
    + permits_2024["units_2"]
    + permits_2024["units_3_4"]
    + permits_2024["units_5_plus"]
)

permits_2024["source_url"] = url_2024

# Validate our seven metropolitan areas.
assert len(permits_2024) == 7, (
    f"Expected 7 MSAs, found {len(permits_2024)}"
)

assert set(permits_2024["cbsa"]) == set(target_cbsas), (
    "Missing or unexpected CBSAs"
)

assert permits_2024["cbsa"].is_unique, (
    "Duplicate CBSA identifiers"
)

assert permits_2024["metro_type"].eq(2).all() | permits_2024["metro_type"].eq(4).all() == False, (
    "Check metropolitan classification codes"
)

assert permits_2024[
    numeric_columns_2024 + ["permits_total"]
].notna().all().all(), "Missing housing-unit values"

assert (
    permits_2024[numeric_columns_2024 + ["permits_total"]] >= 0
).all().all(), "Negative housing-unit values"

print("2024 TARGET MSA VALIDATION")
print("=" * 60)
print("MSAs found:", len(permits_2024))
print("Unique CBSAs:", permits_2024["cbsa"].nunique())
print("Metro classification codes:",
      sorted(permits_2024["metro_type"].unique()))

display(
    permits_2024[
        ["year", "cbsa", "metro_name", "permits_total"]
    ].sort_values("cbsa")
)


2024 TARGET MSA VALIDATION
MSAs found: 7
Unique CBSAs: 7
Metro classification codes: [2, 4]


,year,cbsa,metro_name,permits_total
628,2024,16700,Charleston-North Charleston SC,8514
135,2024,19100,Dallas-Fort Worth-Arlington TX,71788
275,2024,28140,Kansas City MO-KS,9148
300,2024,29820,Las Vegas-Henderson-North Las Vegas NV,14754
377,2024,34980,Nashville-Davidson--Murfreesboro--Franklin TN,20038
398,2024,35620,New York-Newark-Jersey City NY-NJ,57929
490,2024,41860,San Francisco-Oakland-Fremont CA,5914


In [44]:

# Consolidate the 2005–2024 Building Permits datasets.
# Keep a consistent set of columns across source formats.

dataset_groups = [
    permits_2005,
    permits_2006,
    permits_2007_2009,
    permits_2010,
    permits_2011_2013,
    parsed_2014_versions["New-universe 2014"],
    permits_2015_2017,
    permits_2018,
    permits_2019,
    permits_2020_2021,
    permits_2022_2023,
    permits_2024
]

standard_columns = [
    "year",
    "cbsa",
    "metro_name",
    "permits_total",
    "units_1",
    "units_2",
    "units_3_4",
    "units_5_plus",
    "structures_5_plus"
]

standardized = []

for df in dataset_groups:
    subset = df[standard_columns].copy()

    subset["year"] = pd.to_numeric(
        subset["year"], errors="raise"
    ).astype(int)

    subset["cbsa"] = (
        subset["cbsa"].astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    for column in standard_columns[3:]:
        subset[column] = pd.to_numeric(
            subset[column], errors="raise"
        )

    standardized.append(subset)

permits_panel = pd.concat(
    standardized,
    ignore_index=True
)

# Verify expected year and CBSA coverage.
expected_pairs = {
    (year, cbsa)
    for year in range(2005, 2025)
    for cbsa in target_cbsas
}

actual_pairs = set(
    zip(permits_panel["year"], permits_panel["cbsa"])
)

assert len(permits_panel) == 140, (
    f"Expected 140 observations, found {len(permits_panel)}"
)

assert actual_pairs == expected_pairs, (
    "Missing or unexpected CBSA-year combinations"
)

assert not permits_panel.duplicated(
    ["year", "cbsa"]
).any(), "Duplicate CBSA-year combinations"

print("BUILDING PERMITS PANEL VALIDATION")
print("=" * 60)
print("Total observations:", len(permits_panel))
print("Unique years:", permits_panel["year"].nunique())
print("Unique CBSAs:", permits_panel["cbsa"].nunique())
print(
    "Duplicate CBSA-year pairs:",
    permits_panel.duplicated(["year", "cbsa"]).sum()
)
print(
    "Missing CBSA-year pairs:",
    len(expected_pairs - actual_pairs)
)

display(
    permits_panel.pivot(
        index="year",
        columns="cbsa",
        values="permits_total"
    ).rename(columns=target_cbsas)
)


In [45]:

# Inspect historical dataset structures before consolidation.

dataset_names = [
    "2005",
    "2006",
    "2007–2009",
    "2010",
    "2011–2013",
    "2014 New Universe",
    "2015–2017",
    "2018",
    "2019",
    "2020–2021",
    "2022–2023",
    "2024"
]

for name, dataset in zip(dataset_names, dataset_groups):

    print(f"\n{'=' * 65}")
    print(f"DATASET: {name}")
    print(f"Python type: {type(dataset).__name__}")

    if isinstance(dataset, pd.DataFrame):
        print(f"Shape: {dataset.shape}")
        print(f"Columns: {dataset.columns.tolist()}")
        display(dataset.head(2))

    elif isinstance(dataset, list):
        print(f"Records: {len(dataset)}")

        if dataset:
            print(f"First record: {dataset[0]}")

    else:
        print("Unexpected dataset type")



DATASET: 2005
Python type: DataFrame
Shape: (361, 10)
Columns: ['metro_name', 'year', 'permits_total', 'units_1', 'units_2', 'units_3_4', 'units_5_plus', 'structures_5_plus', 'calculated_total', 'total_difference']


,metro_name,year,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus,calculated_total,total_difference
0,"Abilene, TX",2005,346,310,22,0,14,1,346,0
1,"Akron, OH",2005,2724,2310,98,105,211,14,2724,0



DATASET: 2006
Python type: DataFrame
Shape: (7, 10)
Columns: ['year', 'cbsa', 'metro_name', 'permits_total', 'units_1', 'units_2', 'units_3_4', 'units_5_plus', 'structures_5_plus', 'components_sum']


,year,cbsa,metro_name,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus,components_sum
0,2006,16700,"Charleston-North Charleston,* SC",9073,7304,58,84,1627,117,9073
1,2006,19100,"Dallas-Fort Worth-Arlington,* TX",56514,43591,596,329,11998,456,56514



DATASET: 2007–2009
Python type: DataFrame
Shape: (21, 9)
Columns: ['year', 'cbsa', 'metro_name', 'permits_total', 'units_1', 'units_2', 'units_3_4', 'units_5_plus', 'structures_5_plus']


,year,cbsa,metro_name,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus
0,2007,16700,"Charleston-North Charleston,* SC",6780,5602,12,18,1148,60
1,2007,19100,"Dallas-Fort Worth-Arlington,* TX",43568,28156,372,532,14508,490


... (14 outputs left out)


DATASET: 2022–2023
Python type: DataFrame
Shape: (14, 11)
Columns: ['CSA', 'cbsa', 'metro_name', 'permits_total', 'units_1', 'units_2', 'units_3_4', 'units_5_plus', 'structures_5_plus', 'year', 'source_url']


,CSA,cbsa,metro_name,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus,year,source_url
0,999.0,16700,"Charleston-North Charleston, SC",9491,6329,2,32,3128,129,2022,https://www.census.gov/construction/bps/xls/msaannual_202299.xls
1,206.0,19100,"Dallas-Fort Worth-Arlington, TX",77927,43685,686,1314,32242,683,2022,https://www.census.gov/construction/bps/xls/msaannual_202299.xls



DATASET: 2024
Python type: DataFrame
Shape: (7, 11)
Columns: ['year', 'cbsa', 'metro_name', 'metro_type', 'units_1', 'units_2', 'units_3_4', 'units_5_plus', 'structures_5_plus', 'permits_total', 'source_url']


,year,cbsa,metro_name,metro_type,units_1,units_2,units_3_4,units_5_plus,structures_5_plus,permits_total,source_url
135,2024,19100,Dallas-Fort Worth-Arlington TX,2,46440,1500,698,23150,624,71788,https://www2.census.gov/econ/bps/CBSA%20%28beginning%20Jan%202024%29/cbsa2024a.txt
275,2024,28140,Kansas City MO-KS,2,4875,248,630,3395,106,9148,https://www2.census.gov/econ/bps/CBSA%20%28beginning%20Jan%202024%29/cbsa2024a.txt


In [46]:

# Identify the seven target metropolitan areas in the
# 2005 dataset, which does not contain CBSA identifiers.

target_names_2005 = {
    "16700": "Charleston",
    "19100": "Dallas-Fort Worth",
    "28140": "Kansas City",
    "29820": "Las Vegas",
    "34980": "Nashville",
    "35620": "New York",
    "41860": "San Francisco"
}

matched_records = []

for cbsa, name_prefix in target_names_2005.items():

    matches = permits_2005[
        permits_2005["metro_name"].str.startswith(
            name_prefix,
            na=False
        )
    ].copy()

    print(f"\nCBSA {cbsa} — {name_prefix}")
    print("Matching records:", len(matches))

    display(
        matches[
            ["metro_name", "year", "permits_total"]
        ]
    )

    if len(matches) == 1:
        matches["cbsa"] = cbsa
        matched_records.append(matches)

# Only create the dataset if every match is unique.
if len(matched_records) == 7:
    permits_2005_targets = pd.concat(
        matched_records,
        ignore_index=True
    )

    print("\n2005 TARGET DATASET CREATED")
    print("Records:", len(permits_2005_targets))
    print(
        "Unique CBSAs:",
        permits_2005_targets["cbsa"].nunique()
    )
else:
    print(
        "\nReview the matches before proceeding."
    )



CBSA 16700 — Charleston
Matching records: 2


,metro_name,year,permits_total
57,"Charleston, WV",2005,665
58,"Charleston-North Charleston,* SC",2005,10818



CBSA 19100 — Dallas-Fort Worth
Matching records: 1


,metro_name,year,permits_total
80,"Dallas-Fort Worth-Arlington,* TX",2005,59895



CBSA 28140 — Kansas City
Matching records: 1


,metro_name,year,permits_total
168,"Kansas City,* MO-KS",2005,15218


... (5 outputs left out)

,metro_name,year,permits_total
232,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",2005,67207



CBSA 41860 — San Francisco
Matching records: 1


,metro_name,year,permits_total
290,"San Francisco-Oakland-Fremont,* CA",2005,14883



Review the matches before proceeding.


In [47]:

# Match the seven 2005 metropolitan records using
# distinctive name and state patterns.

target_patterns_2005 = {
    "16700": r"^Charleston-North Charleston.*SC$",
    "19100": r"^Dallas-Fort Worth.*TX$",
    "28140": r"^Kansas City.*MO-KS$",
    "29820": r"^Las Vegas.*NV$",
    "34980": r"^Nashville.*TN$",
    "35620": r"^New York.*NY-NJ-PA$",
    "41860": r"^San Francisco.*CA$"
}

matched_records = []

for cbsa, pattern in target_patterns_2005.items():

    matches = permits_2005[
        permits_2005["metro_name"].str.contains(
            pattern,
            regex=True,
            na=False
        )
    ].copy()

    assert len(matches) == 1, (
        f"CBSA {cbsa}: Expected 1 match, "
        f"found {len(matches)}"
    )

    matches["cbsa"] = cbsa
    matched_records.append(matches)

permits_2005_targets = pd.concat(
    matched_records,
    ignore_index=True
)

assert len(permits_2005_targets) == 7
assert permits_2005_targets["cbsa"].nunique() == 7
assert set(permits_2005_targets["cbsa"]) == set(target_cbsas)

print("2005 TARGET MSA VALIDATION")
print("=" * 60)
print("Records:", len(permits_2005_targets))
print("Unique CBSAs:", permits_2005_targets["cbsa"].nunique())
print("Duplicate CBSAs:",
      permits_2005_targets["cbsa"].duplicated().sum())

display(
    permits_2005_targets[
        ["year", "cbsa", "metro_name", "permits_total"]
    ].sort_values("cbsa")
)


2005 TARGET MSA VALIDATION
Records: 7
Unique CBSAs: 7
Duplicate CBSAs: 0


,year,cbsa,metro_name,permits_total
0,2005,16700,"Charleston-North Charleston,* SC",10818
1,2005,19100,"Dallas-Fort Worth-Arlington,* TX",59895
2,2005,28140,"Kansas City,* MO-KS",15218
3,2005,29820,"Las Vegas-Paradise,* NV",39237
4,2005,34980,"Nashville-Davidson--Murfreesboro,* TN",16654
5,2005,35620,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",67207
6,2005,41860,"San Francisco-Oakland-Fremont,* CA",14883


In [48]:

# Consolidate and validate the full 2005–2024
# Census Building Permits panel.

dataset_groups = [
    permits_2005_targets,
    permits_2006,
    permits_2007_2009,
    permits_2010,
    permits_2011_2013,
    parsed_2014_versions["New-universe 2014"],
    permits_2015_2017,
    permits_2018,
    permits_2019,
    permits_2020_2021,
    permits_2022_2023,
    permits_2024
]

standard_columns = [
    "year", "cbsa", "metro_name", "permits_total",
    "units_1", "units_2", "units_3_4",
    "units_5_plus", "structures_5_plus"
]

numeric_columns = [
    "permits_total", "units_1", "units_2",
    "units_3_4", "units_5_plus", "structures_5_plus"
]

standardized = []

for dataset in dataset_groups:

    df = dataset.copy()

    # Standardize CBSA identifiers.
    df["cbsa"] = (
        pd.to_numeric(df["cbsa"], errors="raise")
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    # Retain only our seven target metropolitan areas.
    df = df[df["cbsa"].isin(target_cbsas)].copy()

    # Retain consistent analytical columns.
    df = df[standard_columns].copy()

    df["year"] = pd.to_numeric(
        df["year"], errors="raise"
    ).astype(int)

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column], errors="raise"
        )

    standardized.append(df)

permits_panel = pd.concat(
    standardized,
    ignore_index=True
)

# Verify full MSA-year coverage.
expected_pairs = {
    (year, cbsa)
    for year in range(2005, 2025)
    for cbsa in target_cbsas
}

actual_pairs = set(
    zip(permits_panel["year"], permits_panel["cbsa"])
)

assert len(permits_panel) == 140, (
    f"Expected 140 observations, found {len(permits_panel)}"
)

assert actual_pairs == expected_pairs, (
    "Missing or unexpected MSA-year combinations"
)

assert not permits_panel.duplicated(
    ["year", "cbsa"]
).any(), "Duplicate MSA-year records"

assert permits_panel[numeric_columns].notna().all().all(), (
    "Missing numeric values"
)

assert (permits_panel[numeric_columns] >= 0).all().all(), (
    "Negative numeric values"
)

assert (
    permits_panel[numeric_columns] % 1 == 0
).all().all(), "Non-integer values"

# Check housing-unit arithmetic.
calculated_total = (
    permits_panel["units_1"]
    + permits_panel["units_2"]
    + permits_panel["units_3_4"]
    + permits_panel["units_5_plus"]
)

assert (
    permits_panel["permits_total"] == calculated_total
).all(), "Housing-unit arithmetic mismatch"

permits_panel[numeric_columns] = (
    permits_panel[numeric_columns].astype("int64")
)

permits_panel = permits_panel.sort_values(
    ["year", "cbsa"]
).reset_index(drop=True)

print("FINAL BUILDING PERMITS PANEL VALIDATION")
print("=" * 60)
print("Total observations:", len(permits_panel))
print("Years:", permits_panel["year"].min(),
      "through", permits_panel["year"].max())
print("Unique years:", permits_panel["year"].nunique())
print("Unique CBSAs:", permits_panel["cbsa"].nunique())
print("Duplicate MSA-year records:",
      permits_panel.duplicated(["year", "cbsa"]).sum())
print("Missing MSA-year combinations:",
      len(expected_pairs - actual_pairs))
print("Housing-unit arithmetic mismatches:",
      (permits_panel["permits_total"] != calculated_total).sum())

print("\nHOUSING UNITS AUTHORIZED BY YEAR AND MSA")

display(
    permits_panel.pivot(
        index="year",
        columns="cbsa",
        values="permits_total"
    ).rename(columns=target_cbsas)
)


FINAL BUILDING PERMITS PANEL VALIDATION
Total observations: 140
Years: 2005 through 2024
Unique years: 20
Unique CBSAs: 7
Duplicate MSA-year records: 0
Missing MSA-year combinations: 0
Housing-unit arithmetic mismatches: 7

HOUSING UNITS AUTHORIZED BY YEAR AND MSA


cbsa,"Charleston–North Charleston, SC","Dallas–Fort Worth, TX","Kansas City, MO–KS","Las Vegas, NV","Nashville, TN","New York, NY","San Francisco, CA"
year,,,,,,,
2005,10818,59895,15218,39237,16654,67207,14883
2006,9073,56514,13205,33728,15327,60987,14483
2007,6780,43568,8129,24089,13567,56405,10560
2008,5172,36321,5300,12537,8188,51590,7555
2009,2949,20370,3406,5688,4933,16707,3550
2010,3060,19558,2714,5474,5092,18668,4621
2011,3822,24827,3287,5147,5394,21539,5783
2012,4593,35042,4981,7375,8247,26912,9163
2013,5417,37910,7532,8573,10889,39824,10922


In [49]:

# Revalidate housing-unit arithmetic after consolidation
# using the final sorted DataFrame.

permits_panel["calculated_total"] = (
    permits_panel["units_1"]
    + permits_panel["units_2"]
    + permits_panel["units_3_4"]
    + permits_panel["units_5_plus"]
)

permits_panel["total_difference"] = (
    permits_panel["permits_total"]
    - permits_panel["calculated_total"]
)

mismatches = permits_panel[
    permits_panel["total_difference"] != 0
]

print("FINAL ARITHMETIC REVALIDATION")
print("=" * 60)
print("Total observations:", len(permits_panel))
print("Arithmetic mismatches:", len(mismatches))
print("Maximum absolute difference:",
      permits_panel["total_difference"].abs().max())

if mismatches.empty:
    print("\nPASS: All 140 housing-unit totals reconcile.")
else:
    print("\nREVIEW REQUIRED: Mismatched records")
    display(
        mismatches[
            [
                "year", "cbsa", "metro_name",
                "permits_total", "calculated_total",
                "total_difference"
            ]
        ]
    )

assert mismatches.empty, (
    "Housing-unit totals do not reconcile."
)


FINAL ARITHMETIC REVALIDATION
Total observations: 140
Arithmetic mismatches: 0
Maximum absolute difference: 0

PASS: All 140 housing-unit totals reconcile.


In [50]:

# Export the validated 2005–2024 Building Permits panel
# to the MIS581 Capstone data directory.

from pathlib import Path

# Capstone data directory
output_dir = Path(
    r"../data"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Keep fields needed for downstream integration.
export_columns = [
    "year", "cbsa", "metro_name", "permits_total",
    "units_1", "units_2", "units_3_4",
    "units_5_plus", "structures_5_plus"
]

export_df = permits_panel[export_columns].copy()

# Export the consolidated dataset.
data_path = (
    output_dir / "building_permits_msa_2005_2024.csv"
)

export_df.to_csv(data_path, index=False)

# Create a validation summary.
validation_summary = pd.DataFrame([{
    "dataset": "Census Building Permits MSA",
    "start_year": int(export_df["year"].min()),
    "end_year": int(export_df["year"].max()),
    "observations": len(export_df),
    "unique_years": export_df["year"].nunique(),
    "unique_cbsas": export_df["cbsa"].nunique(),
    "duplicate_msa_years": int(
        export_df.duplicated(["year", "cbsa"]).sum()
    ),
    "arithmetic_mismatches": int(
        (permits_panel["total_difference"] != 0).sum()
    )
}])

# Export the validation summary.
validation_path = (
    output_dir / "building_permits_validation_summary.csv"
)

validation_summary.to_csv(validation_path, index=False)

print("FILES EXPORTED SUCCESSFULLY")
print("=" * 60)
print("Dataset:", data_path)
print("Validation summary:", validation_path)

print("\nVALIDATION SUMMARY")
display(validation_summary)

print("\nEXPORTED DATASET PREVIEW")
display(export_df.head(7))


FILES EXPORTED SUCCESSFULLY
Dataset: <LOCAL_FILE_PATH>
Validation summary: <LOCAL_FILE_PATH>

VALIDATION SUMMARY


,dataset,start_year,end_year,observations,unique_years,unique_cbsas,duplicate_msa_years,arithmetic_mismatches
0,Census Building Permits MSA,2005,2024,140,20,7,0,0



EXPORTED DATASET PREVIEW


,year,cbsa,metro_name,permits_total,units_1,units_2,units_3_4,units_5_plus,structures_5_plus
0,2005,16700,"Charleston-North Charleston,* SC",10818,8084,156,129,2449,155
1,2005,19100,"Dallas-Fort Worth-Arlington,* TX",59895,49489,442,152,9812,509
2,2005,28140,"Kansas City,* MO-KS",15218,12118,528,749,1823,109
3,2005,29820,"Las Vegas-Paradise,* NV",39237,30479,96,1155,7507,348
4,2005,34980,"Nashville-Davidson--Murfreesboro,* TN",16654,14074,322,75,2183,123
5,2005,35620,"New York-Northern New Jersey-Long Island,* NY-NJ-PA",67207,21424,6900,6570,32313,1163
6,2005,41860,"San Francisco-Oakland-Fremont,* CA",14883,8102,78,294,6409,188
